# Prompt Iteration — Pairwise (A/B) Evaluation & Align Evals for the Platform Docs Assistant

This module iterates on the **platform docs assistant** (`src/pov_agent/agent.py`), the same
agent as the [evaluation module](./langsmith_walkthrough.ipynb). The notebook is
**self-contained**: the *Standalone setup* cells rebuild everything it needs, so you can run
it top to bottom on a fresh kernel.

The evaluation module scores each experiment **absolutely** (correctness, fairness,
citations) and gates CI on it. This module adds two iteration tools:

1. **Pairwise (A/B) evaluation.** When you're choosing between two candidates (a new
   model, a reworded prompt), the question is *relative*: which answer is better? An LLM
   judge sees both answers side by side and picks a winner.
2. **Align Evals.** Tune a subjective, house-style judge (here, **runbook-ready**) to agree
   with a human expert before you trust it.

**Prerequisites:** a `.env` with `LANGSMITH_API_KEY`, `LANGSMITH_TRACING=true`, and the
provider key for `MODEL`, `JUDGE_MODEL`, and `CANDIDATE_MODEL`.

## Standalone setup

### Setup 1 — environment, client, and the agent

In [ ]:
import os
import sys
import uuid
import random
import logging
from pathlib import Path

here = Path().resolve()
src_dir = None
for candidate in (here, *here.parents):
    if (candidate / "pov_agent" / "agent.py").exists():
        src_dir = candidate
        break
    if (candidate / "src" / "pov_agent" / "agent.py").exists():
        src_dir = candidate / "src"
        break
if src_dir is None:
    raise RuntimeError("Could not locate the pov_agent package on any parent path.")
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from dotenv import load_dotenv
load_dotenv(dotenv_path=src_dir.parent / ".env", override=True)

from langsmith import Client

class _NoIndexedRunIDFilter(logging.Filter):
    def filter(self, record):
        return "No indexed run ID" not in record.getMessage()

logging.getLogger("langchain_core.callbacks.manager").addFilter(_NoIndexedRunIDFilter())

from pov_agent.agent import ask, build_agent
from pov_agent.config import DATASET_NAME, DEFAULT_MODEL, get_judge_model

client = Client()
PROJECT = os.environ.get("LANGSMITH_PROJECT", "pov-docs-assistant")
RUN_TAG = uuid.uuid4().hex[:6]
scoped = lambda name: f"{name}-{RUN_TAG}"
judge_model = get_judge_model()

# The shipping agent (v1). local_context=True reads the skill + memory from a throwaway
# local copy, so the notebook works before the Context Hub is bootstrapped.
agent = build_agent("v1", local_context=True)
print("Agent ready on", os.environ.get("MODEL", DEFAULT_MODEL), "| project:", PROJECT)

### Setup 2 — the dataset and the review queue

The same golden dataset the CI gate uses (synced from `seed_data.py`) and the
expert-review queue (`ensure_queue` is idempotent).

In [ ]:
from pov_agent.langsmith.annotation_queues import ensure_queue
from pov_agent.langsmith.datasets import populate_dataset

dataset_name = DATASET_NAME
populate_dataset(client, dataset_name)

queue = ensure_queue(client, run_ids=[])
print(f"Dataset '{dataset_name}' ready; review queue '{queue.name}' ready.")

## Pairwise evaluation (A/B, LLM-as-judge)

Absolute scores answer "is this good enough to ship?" When you're choosing **between**
two candidates, the question is **relative**: *which answer is better?* Pairwise
evaluation shows both answers to a judge **side by side** and asks for a winner. That
sidesteps the calibration problem where an absolute judge passes everything but can still
reliably say *A beats B*.

**The decision: is a model swap worth it?** The most common A/B for a docs assistant is a
**model migration**. A newer or cheaper model ships, and before rolling it out you need to
know whether answers get *better, worse, or indistinguishable*. Candidate **A** is the
agent on the **incumbent model** (`MODEL`), and candidate **B** is the *same agent* (same
prompt, tools, skill, and memory) on a **candidate model** (`CANDIDATE_MODEL`).

**In the UI:** select two experiments in *Datasets & Experiments*, click **Compare**, and
add a pairwise evaluator. No code needed.

### Candidate A — the incumbent model

We score each candidate **absolutely** with the same prebuilt openevals correctness judge
the CI gate uses, so you get each model's correctness alongside the head-to-head win rate.

In [ ]:
from pov_agent.langsmith.evaluators import correctness_evaluator

def make_target(a):
    def target(inputs: dict) -> dict:
        return {"answer": ask(a, inputs["question"])}
    return target

results = client.evaluate(
    make_target(agent), data=dataset_name, evaluators=[correctness_evaluator],
    experiment_prefix=scoped("docs-incumbent"), max_concurrency=4,
)
print(f"Candidate A experiment: {results.experiment_name}")

### Candidate B — the same agent on a candidate model

`build_agent()` reads the `MODEL` env var, so we swap it only while building the
candidate. Set `CANDIDATE_MODEL` in `.env` to the model you're considering (a newer
release, a cheaper tier, or another provider, such as `anthropic:claude-haiku-4-5`).

In [ ]:
INCUMBENT_MODEL = os.environ.get("MODEL", DEFAULT_MODEL)
CANDIDATE_MODEL = os.environ.get("CANDIDATE_MODEL", "openai:gpt-5-nano")
print(f"Incumbent (A): {INCUMBENT_MODEL}\nCandidate (B): {CANDIDATE_MODEL}")

_saved = os.environ.get("MODEL")
os.environ["MODEL"] = CANDIDATE_MODEL
try:
    candidate_agent = build_agent("v1", local_context=True)
finally:
    if _saved is None:
        os.environ.pop("MODEL", None)
    else:
        os.environ["MODEL"] = _saved

candidate_results = client.evaluate(
    make_target(candidate_agent), data=dataset_name, evaluators=[correctness_evaluator],
    experiment_prefix=scoped("docs-candidate-model"), max_concurrency=4,
    metadata={"model": CANDIDATE_MODEL},
)
print(f"Candidate B experiment: {candidate_results.experiment_name}")

Now the pairwise judge, the **comparative twin** of correctness. It sees the question, the
reference answer from the docs, and **both** answers, with the order randomized to avoid
position bias, and picks the one that is more correct, complete, and actionable for that
user. It doesn't know which model wrote which answer. A comparative evaluator receives
the two `runs` and the `example`, and returns `scores` keyed by run ID (`1` for the
winner, `0` for the loser, `0.5` each on a tie). LangSmith aggregates those into a
**win rate**.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage

class PairwiseGrade(BaseModel):
    preferred: int = Field(description="1 if Answer A is better, 2 if Answer B is better, 0 if tie.")
    reasoning: str = Field(description="One-sentence explanation.")

PAIRWISE_PROMPT = """You compare two answers from a platform docs assistant to the same user question.

Prefer the answer that:
- is factually consistent with the REFERENCE ANSWER (taken from the docs) and invents nothing;
- covers everything the user asked, with the concrete steps, UI paths, and commands they need;
- is actionable for this user as written, and cites its sources.

An answer that invents menus, commands, or paths, skips what the user asked, or talks down to the user should lose.
Return preferred=1 if A is better, 2 if B is better, or 0 only if they are genuinely equivalent."""

pairwise_judge = judge_model.with_structured_output(PairwiseGrade)

def pairwise_evaluator(runs: list, example) -> dict:
    a, b = random.sample(runs, 2)          # shuffle so the judge can't exploit position
    grade: PairwiseGrade = pairwise_judge.invoke([
        SystemMessage(content=PAIRWISE_PROMPT),
        HumanMessage(content=(
            f"Question: {example.inputs['question']}\n\n"
            f"Reference answer: {(example.outputs or {}).get('answer', '')}\n\n"
            f"Answer A:\n{(a.outputs or {}).get('answer')}\n\n"
            f"Answer B:\n{(b.outputs or {}).get('answer')}"
        )),
    ])
    scores = {a.id: 0.5, b.id: 0.5}
    if grade.preferred == 1:
        scores = {a.id: 1, b.id: 0}
    elif grade.preferred == 2:
        scores = {a.id: 0, b.id: 1}
    return {"key": "pairwise_preference", "scores": scores, "comment": grade.reasoning}

### Run the comparison

Comparative mode takes the **two experiment names** and the comparative evaluator. The
compare view shows per-example winners and the overall **win rate**. Read that alongside
each model's correctness, cost, and latency: that's the evidence for or against migrating.

In [ ]:
comparison = client.evaluate(
    [results.experiment_name, candidate_results.experiment_name],
    evaluators=[pairwise_evaluator],
    experiment_prefix=scoped("docs-pairwise"),
    max_concurrency=4,
)
print("Pairwise comparison complete — open the compare view in LangSmith to see win rates.")

## Align Evals — tune a judge to a human expert

The prebuilt correctness and fairness judges work out of the box because the model already
knows what "correct" and "unequal treatment" mean. The judgments worth **aligning** are the
**subjective, house-style** ones the model has no prior about.

For our platform team that's **runbook-ready**. The team's standard says an answer to a
"how do I…" question is ready to follow only if it has **all four** of:
1. a **direct answer** up front,
2. **numbered steps** with the exact UI path, command, or file path,
3. **how to confirm it worked** (what the user should see, such as a status of *Ready* or
   *Running*, or an endpoint URL), and
4. a **`Sources:`** line.

An answer that nails three of the four is *not* runbook-ready, even though a model left to
its own devices will happily call it complete.

**Align Evals** closes that gap. You label real outputs the way the expert would, then
iterate the judge prompt in the **Evaluator Playground** until its scores match your
labels. The metric is the **alignment score**, the share of examples where the judge
agrees with the human.

📖 [Improve LLM-as-judge evaluators using human feedback →](https://docs.langchain.com/langsmith/improve-judge-evaluator-feedback)

### 1. Select runs — seed the queue with answers to label

We generate a balanced set: full answers (likely runbook-ready) and answers nudged toward
a partial form (likely not). Then we add exactly those runs to the review queue. (To keep
feeding the queue from live traffic, use a routing rule like the one in the evaluation
module's Part 5.)

In [ ]:
align_requests = [
    ("How do I deploy a Llama model from Hugging Face behind an API endpoint on Nimbus AI Platform?", ""),
    ("How do I create a notebook with a GPU in Nimbus Workbench?", ""),
    ("How do I import our own Helm chart as a framework?", ""),
    ("How do I deploy a Llama model from Hugging Face behind an API endpoint on Nimbus AI Platform?", " Just one sentence, please."),
    ("How do I create a notebook with a GPU in Nimbus Workbench?", " Skip the steps, just tell me where to go."),
    ("How do I import our own Helm chart as a framework?", " Only list what the chart needs, not the import steps."),
]
import time
from datetime import datetime, timedelta, timezone
from langchain_core.tracers.langchain import wait_for_all_tracers

align_tag = scoped("align")
for question, nudge in align_requests:
    ask(agent, question + nudge, metadata={"source": "notebook"}, tags=["notebook", align_tag])
wait_for_all_tracers()
print(f"Generated {len(align_requests)} answers.")

# Add exactly these runs to the review queue (ingestion lags a few seconds).
since = datetime.now(timezone.utc) - timedelta(minutes=30)
for _ in range(10):
    align_runs = list(client.list_runs(project_name=PROJECT, is_root=True, start_time=since,
                                       filter=f'has(tags, "{align_tag}")'))
    if len(align_runs) >= len(align_requests):
        break
    time.sleep(3)
client.add_runs_to_annotation_queue(queue.id, run_ids=[r.id for r in align_runs])
print(f"Added {len(align_runs)} run(s) to queue '{queue.name}'.")

### 2. Label the runs (in the annotation queue)

Open **Annotation Queues → `pov-docs-answer-review`** and score each answer the way the
platform team's expert would:

- Score **1** only if it has **all four**: direct answer, numbered steps with exact paths
  or commands, how to confirm it worked, and a `Sources:` line.
- Score **0** if any one is missing, even if the rest is accurate and well written.

After scoring, click **Add to Reference Dataset**. Aim for **~20 examples** with a balance
of 0s and 1s; re-run the seed cell with new questions if you need more.

### 3. Test the judge prompt in the Evaluator Playground

1. **Datasets & Experiments** → your reference dataset → **+ Evaluator** → **Create from labeled data**.
2. Name the feedback key **`runbook_ready`**.
3. Paste the starter prompt below, enable **reasoning**, and click **Start Alignment**.

```text
You are grading an answer from a Nimbus AI Platform docs assistant against the platform
team's house style.

An answer is RUNBOOK-READY (score 1) only if it has ALL FOUR of:
  1. A direct answer in the first one or two sentences.
  2. Numbered steps that name the exact UI path, command, or file path to use.
  3. How to confirm it worked — what the user should see when it succeeded
     (e.g. a status of Ready/Running, an endpoint URL, a file in /mnt/shared).
  4. A final "Sources:" line naming the docs used.

If ANY of the four is missing, score 0 — even if what is present is accurate.

Return score=1 or score=0 and one sentence naming which of the four are present and missing.
```

Expect the judge to start out too lenient. It tends to count "click Launch" as a
confirmation step, and that's exactly the gap alignment closes.

### 4. Refine & repeat until the judge agrees with you

- **Investigate misaligned examples.** Read the judge's reasoning, group the misses into
  failure modes, and add an explicit instruction for each (for example: "the final step of
  a procedure is not a confirmation step").
- **Pin down edge cases.** For a "why is X happening" question, decide whether "how to
  confirm" means "how to confirm the cause", then write that into the rubric.
- **Add labels, then re-check**, so you don't overfit the first handful.

When the alignment score is high enough to trust, promote `runbook_ready` to an **online
evaluator** (Part 4 of the evaluation module) or add it to the CI gate's
`CI_THRESHOLDS`.

## Teardown

In [ ]:
# Nothing to delete by default: the review queue holds your labels, and experiments and
# traces are history. To remove the queue's labeled runs, delete them in the UI.
print("Experiments, traces, and the review queue are kept (the queue holds your labels).")